# CelebA compositional retrieval with CLIP: Arithmetic, SLERP, CoOp, and NOAH

This notebook refactors the previous experimental notebooks into one more modular workflow. The goal is to evaluate several ways of turning a **reference face image** plus a **signed attribute query** into a retrieval vector in CLIP space.

Example query:

```text
+Smiling, -Eyeglasses
```

means: start from the reference image, move toward the concept *Smiling*, move away from the concept *Eyeglasses*, and retrieve the most similar CelebA images.

The notebook intentionally separates the pipeline into reusable blocks:

1. dataset and annotation loading;
2. CLIP image/text feature extraction;
3. the unchanged shared metric function;
4. case construction, visualization, and experiment runners;
5. zero-shot methods: latent arithmetic and SLERP;
6. trainable methods: CoOp prompt learning and NOAH-style searchable adapters.


## Method motivation

### Latent-space arithmetic

CLIP maps images and text into a shared vector space. The simplest retrieval assumption is that attribute edits behave approximately like vector directions:

```python
target = normalize(reference_image + scale * query_direction)
```

This is cheap and zero-shot, but it assumes the embedding space is locally linear for facial attributes.

### SLERP

CLIP features are normally compared with cosine similarity after normalization. SLERP, or spherical linear interpolation, respects this unit-sphere geometry better than plain addition. It interpolates between the reference-image direction and the text-query direction on the sphere.

### CoOp

CoOp, from **Learning to Prompt for Vision-Language Models** by Zhou et al., observes that CLIP is sensitive to prompt wording. Instead of manually choosing prompts, CoOp learns continuous context vectors while keeping CLIP frozen:  
https://arxiv.org/abs/2109.01134

In this notebook, CoOp is adapted from classification to retrieval: the learned prompt creates better attribute text directions, and those learned directions are used in the same reference-plus-query retrieval formula.

### NOAH-style adapters

NOAH, from **Neural Prompt Search** by Zhang et al., frames parameter-efficient adaptation design as a search problem: different small prompt/adaptation modules and sizes can work better on different datasets.  
https://arxiv.org/abs/2206.04673

The implementation here is not a transformer-block NOAH reproduction. It is a retrieval-specific adaptation of the NOAH idea: a searchable supernet of lightweight residual bottleneck adapters is placed around the reference embedding, query direction, and fused target embedding. Candidate adapter sizes and text scales are evaluated with the same retrieval metric.


## 1. Setup

The notebook is written for Google Colab. The expensive operations are optional or limited by default; use the final template cells only when you are ready for longer runs.


In [ ]:
%pip install -q torch torchvision tensorboard ftfy regex tqdm scikit-learn scikit-image transformers huggingface_hub


In [ ]:
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KS = [1, 5, 10]
DEFAULT_MAX_REFERENCES = 10

print(f"Using device: {DEVICE}")


## 2. Dataset and annotations

Expected Drive layout by default:

```text
/content/drive/MyDrive/datasets/celeba.zip
/content/drive/MyDrive/datasets/celeba_evaluation.json
/content/drive/MyDrive/datasets/celeba_image_embeddings.pt
```

Change the paths in the next cell if your files live elsewhere.


In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")

DATA_ROOT = Path("/content/datasets")
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/datasets")
CELEBA_ZIP_PATH = DRIVE_DATA_ROOT / "celeba.zip"
ANNOTATIONS_PATH = DRIVE_DATA_ROOT / "celeba_evaluation.json"
LOCAL_EMBEDDINGS_PATH = Path("/content/celeba_image_embeddings.pt")
DRIVE_EMBEDDINGS_PATH = DRIVE_DATA_ROOT / "celeba_image_embeddings.pt"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB and CELEBA_ZIP_PATH.exists() and not (DATA_ROOT / "celeba").exists():
    # The -n flag avoids overwriting files if the archive was already extracted.
    import subprocess
    subprocess.run(["unzip", "-n", "-q", str(CELEBA_ZIP_PATH), "-d", str(DATA_ROOT)], check=True)


In [ ]:
from torchvision.datasets import CelebA

celeba = CelebA(root=DATA_ROOT, split="test", download=False)
print(f"Loaded CelebA test split. Dataset length: {len(celeba)}")
print(f"Number of CelebA attributes: {len(celeba.attr_names)}")


In [ ]:
with open(ANNOTATIONS_PATH, "r") as f:
    annotations = json.load(f)

print(f"Loaded {len(annotations)} evaluation queries from {ANNOTATIONS_PATH}")
print("First query:", annotations[0]["query"])


In [ ]:
def show_celeba_examples(indices=(0, 1, 2, 3, 4)):
    fig, axes = plt.subplots(1, len(indices), figsize=(2.2 * len(indices), 2.5))
    if len(indices) == 1:
        axes = [axes]

    for axis, image_index in zip(axes, indices):
        img, _ = celeba[int(image_index)]
        axis.imshow(img)
        axis.set_title(f"ID {image_index}")
        axis.axis("off")

    plt.tight_layout()
    plt.show()

show_celeba_examples()


## 3. CLIP model and frozen image embeddings

We keep CLIP frozen for every method. Arithmetic and SLERP are fully zero-shot. CoOp and NOAH only train small additional parameters.


In [ ]:
from transformers import CLIPModel, CLIPProcessor
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

MODEL_NAME = "openai/clip-vit-base-patch32"
preprocess = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)

print(f"Loaded {MODEL_NAME}")
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Input resolution:", model.config.vision_config.image_size)
print("Text context length:", model.config.text_config.max_position_embeddings)
print("Projection dimension:", model.config.projection_dim)


## Robust CLIP feature extraction

The retrieval methods require image and text features to be plain tensors in the same CLIP projection space. In standard Hugging Face `CLIPModel`, `get_image_features(...)` and `get_text_features(...)` return tensors directly. In some notebook/runtime combinations, however, the returned object may be a pooled model output instead. The helper below normalizes both cases: it extracts the pooled tensor and applies CLIP's projection layer when necessary.


In [ ]:
def normalize_embeddings(embeddings, eps=1e-8):
    """L2-normalize a vector or batch of vectors along the last dimension."""
    return embeddings / (embeddings.norm(dim=-1, keepdim=True) + eps)


def clip_feature_tensor(output, projection_layer=None):
    """Return a tensor from different Hugging Face CLIP output styles.

    In most recent `CLIPModel` versions, `get_image_features` and
    `get_text_features` return tensors directly. Some runtime/model
    combinations instead return a `BaseModelOutputWithPooling` object.
    This helper keeps the notebook robust by extracting the pooled tensor
    and, when needed, applying CLIP's projection layer so image/text
    features stay in the same retrieval space.
    """
    if torch.is_tensor(output):
        return output

    for attribute_name in ("image_embeds", "text_embeds", "pooler_output"):
        value = getattr(output, attribute_name, None)
        if value is not None:
            if projection_layer is not None:
                in_features = getattr(projection_layer, "in_features", None)
                if in_features is not None and value.shape[-1] == in_features:
                    value = projection_layer(value)
            return value

    if isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        value = output[0]
        if projection_layer is not None:
            in_features = getattr(projection_layer, "in_features", None)
            if in_features is not None and value.shape[-1] == in_features:
                value = projection_layer(value)
        return value

    raise TypeError(f"Could not convert CLIP output of type {type(output)} to a tensor.")


@torch.no_grad()
def compute_celeba_image_embeddings(celeba_dataset, model, preprocess, batch_size=64):
    """Compute one CLIP image embedding per CelebA test image."""
    batches = []
    model.eval()
    image_projection = getattr(model, "visual_projection", None)

    for start in tqdm(range(0, len(celeba_dataset), batch_size), desc="Computing image embeddings"):
        images = [celeba_dataset[i][0] for i in range(start, min(start + batch_size, len(celeba_dataset)))]
        inputs = preprocess(images=images, return_tensors="pt")
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

        output = model.get_image_features(**inputs)
        features = clip_feature_tensor(output, projection_layer=image_projection).float()
        batches.append(features.cpu())

    return torch.cat(batches, dim=0)


def load_or_compute_image_embeddings():
    """Prefer local cache, then Drive cache, otherwise compute and save."""
    if LOCAL_EMBEDDINGS_PATH.exists():
        print(f"Loading image embeddings from {LOCAL_EMBEDDINGS_PATH}")
        return torch.load(LOCAL_EMBEDDINGS_PATH, map_location="cpu")

    if DRIVE_EMBEDDINGS_PATH.exists():
        print(f"Loading image embeddings from {DRIVE_EMBEDDINGS_PATH}")
        embeddings = torch.load(DRIVE_EMBEDDINGS_PATH, map_location="cpu")
        torch.save(embeddings, LOCAL_EMBEDDINGS_PATH)
        return embeddings

    print("No cached image embeddings found. Computing them now.")
    embeddings = compute_celeba_image_embeddings(celeba, model, preprocess)
    torch.save(embeddings, LOCAL_EMBEDDINGS_PATH)

    if DRIVE_DATA_ROOT.exists():
        torch.save(embeddings, DRIVE_EMBEDDINGS_PATH)

    return embeddings


image_embeddings = load_or_compute_image_embeddings().float()
IMAGE_EMBEDDINGS_NORMALIZED = normalize_embeddings(image_embeddings)
EMBEDDING_DIM = IMAGE_EMBEDDINGS_NORMALIZED.shape[-1]

print("Image embeddings:", tuple(IMAGE_EMBEDDINGS_NORMALIZED.shape))


## 4. Evaluation metric — unchanged

The function below is intentionally kept unchanged from the shared notebooks. All runners call this exact function through `score_case`.


In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }


## 5. Modular evaluation workflow

A `RetrievalCase` is one concrete test item:

```text
query index + reference image id + list of valid target image ids
```

The helpers below make it easy to run small debug experiments first and larger full-query evaluations later.


In [ ]:
@dataclass(frozen=True)
class RetrievalCase:
    query_index: int
    query_text: str
    reference_image_index: str
    ground_truth: list[int]


def reference_options(query_index, limit=10):
    """Show valid reference image IDs for a query."""
    ground_truth_by_reference = annotations[query_index]["ground_truth"]
    reference_ids = list(ground_truth_by_reference.keys())

    if limit is not None:
        reference_ids = reference_ids[:limit]

    return pd.DataFrame([
        {
            "reference_position": position,
            "reference_image_index": reference_id,
            "num_ground_truth": len(ground_truth_by_reference[reference_id]),
        }
        for position, reference_id in enumerate(reference_ids, start=1)
    ])


def make_case(query_index, reference_image_index=None, reference_position=None):
    """Create one evaluation case for a selected query/reference pair."""
    if reference_image_index is not None and reference_position is not None:
        raise ValueError("Pass either reference_image_index or reference_position, not both.")

    ground_truth_by_reference = annotations[query_index]["ground_truth"]
    reference_ids = list(ground_truth_by_reference.keys())

    if not reference_ids:
        raise ValueError(f"Query {query_index} has no reference images.")

    if reference_image_index is None:
        if reference_position is None:
            reference_position = 1

        if not 1 <= reference_position <= len(reference_ids):
            raise IndexError(
                f"reference_position must be between 1 and {len(reference_ids)} "
                f"for query {query_index}."
            )

        reference_image_index = reference_ids[reference_position - 1]
    else:
        reference_image_index = str(reference_image_index)

    if reference_image_index not in ground_truth_by_reference:
        examples = ", ".join(reference_ids[:10])
        raise ValueError(
            f"Reference image {reference_image_index!r} is not available for query {query_index}. "
            f"Use reference_options({query_index}) to inspect valid IDs. "
            f"First available IDs: {examples}"
        )

    return RetrievalCase(
        query_index=query_index,
        query_text=annotations[query_index]["query"],
        reference_image_index=reference_image_index,
        ground_truth=list(ground_truth_by_reference[reference_image_index]),
    )


def make_cases(query_indices=None, max_references_per_query=DEFAULT_MAX_REFERENCES, reference_ids_by_query=None):
    """
    Build a list of evaluation cases.

    Args:
    ----
        query_indices: list of query indices. None means all queries.
        max_references_per_query: max references per query. Use None for all references.
        reference_ids_by_query: optional dict like {10: ["13", "42"]}.
    """
    if query_indices is None:
        query_indices = range(len(annotations))

    reference_ids_by_query = reference_ids_by_query or {}
    cases = []

    for query_index in query_indices:
        ground_truth_by_reference = annotations[query_index]["ground_truth"]

        if query_index in reference_ids_by_query:
            reference_ids = [str(reference_id) for reference_id in reference_ids_by_query[query_index]]
        else:
            reference_ids = list(ground_truth_by_reference.keys())
            if max_references_per_query is not None:
                reference_ids = reference_ids[:max_references_per_query]

        for reference_id in reference_ids:
            cases.append(make_case(query_index, reference_image_index=reference_id))

    return cases


def split_cases(cases, validation_fraction=0.2, seed=0):
    """Shuffle and split cases into train/validation subsets."""
    cases = list(cases)
    rng = random.Random(seed)
    rng.shuffle(cases)

    split_at = max(1, int(round(len(cases) * (1.0 - validation_fraction))))
    return cases[:split_at], cases[split_at:]


In [ ]:
def score_case(case, predictions, ks=KS):
    """Score one case at several K values using the unchanged metric function."""
    row = {
        "query_index": case.query_index,
        "query": case.query_text,
        "reference_image_index": case.reference_image_index,
        "num_ground_truth": len(case.ground_truth),
    }

    for k in ks:
        metrics = evaluate_retrieval(predictions, case.ground_truth, k)
        row.update(metrics)

    return row


def summarize_results(results):
    """Aggregate per-case metrics into query-level and overall averages."""
    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results)
    metric_columns = [column for column in df.columns if column.startswith(("Recall@", "Precision@"))]

    summary = (
        df.groupby(["query_index", "query"], as_index=False)[metric_columns]
        .mean()
        .sort_values("query_index")
    )

    overall = {"query_index": "ALL", "query": "All selected cases"}
    for column in metric_columns:
        overall[column] = df[column].mean()

    return pd.concat([summary, pd.DataFrame([overall])], ignore_index=True)


def run_retrieval_method(cases, predict_fn, method_name, ks=KS):
    """Generic evaluation loop for any function predict_fn(case, k)."""
    k_max = max(ks)
    rows = []

    for case in tqdm(cases, desc=method_name):
        predictions = predict_fn(case, k_max)
        rows.append(score_case(case, predictions, ks=ks))

    return rows


def compare_summaries(named_results):
    """Create one compact overall table from several result lists."""
    rows = []

    for method_name, results in named_results.items():
        summary = summarize_results(results)
        if summary.empty:
            continue
        overall = summary[summary["query_index"] == "ALL"].iloc[0].to_dict()
        overall["method"] = method_name
        rows.append(overall)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    metric_columns = [column for column in df.columns if column.startswith(("Recall@", "Precision@"))]
    return df[["method"] + metric_columns].sort_values("method")


In [ ]:
def visualize_case(case, predictions, k=5):
    """Show reference image, ground-truth targets, and predictions."""
    print(f"Query {case.query_index}: {case.query_text}")
    print(f"Reference image ID: {case.reference_image_index}")

    reference_img = celeba[int(case.reference_image_index)][0]
    plt.figure(figsize=(3, 3))
    plt.imshow(reference_img)
    plt.title("Reference")
    plt.axis("off")
    plt.show()

    def show_image_row(title, image_indices, title_prefix):
        image_indices = list(image_indices)[:k]
        print(title)

        if not image_indices:
            print("No images to show.")
            return

        cols = min(len(image_indices), 5)
        rows = math.ceil(len(image_indices) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))

        if rows == 1 and cols == 1:
            axes = [axes]
        else:
            axes = axes.flatten()

        for axis, image_index in zip(axes, image_indices):
            img, _ = celeba[int(image_index)]
            axis.imshow(img)
            axis.set_title(f"{title_prefix} {image_index}")
            axis.axis("off")

        for axis in axes[len(image_indices):]:
            axis.axis("off")

        plt.tight_layout()
        plt.show()

    show_image_row(f"Ground truth, first {k}", case.ground_truth, "GT")
    show_image_row(f"Predictions, top {k}", predictions, "Pred")


def inspect_case(case, predictions, ks=KS, k_visualize=5, show_metrics=True, show_images=True):
    """Display metrics and/or images for one case."""
    metrics = score_case(case, predictions, ks=ks)

    if show_metrics:
        display(pd.DataFrame([metrics]))

    if show_images:
        visualize_case(case, predictions, k=k_visualize)

    return metrics


## 6. Text/query embedding helpers

The query parser uses the signed attribute format from the annotation file. Each attribute is embedded once and cached.


In [ ]:
TEXT_EMBEDDING_CACHE = {}
QUERY_DIRECTION_CACHE = {}


@torch.no_grad()
def text_embedding(text):
    """Return a normalized CLIP text embedding on CPU."""
    cache_key = text.strip().lower()
    if cache_key in TEXT_EMBEDDING_CACHE:
        return TEXT_EMBEDDING_CACHE[cache_key]

    processed = preprocess(text=text, return_tensors="pt", padding=True, truncation=True)
    processed = {key: value.to(DEVICE) for key, value in processed.items()}

    output = model.get_text_features(**processed)
    text_projection = getattr(model, "text_projection", None)
    embedding = clip_feature_tensor(output, projection_layer=text_projection).float().squeeze(0).cpu()
    embedding = normalize_embeddings(embedding)
    TEXT_EMBEDDING_CACHE[cache_key] = embedding
    return embedding


def parse_signed_query(query_text):
    """Parse '+Smiling, -Male' into [(+1, 'Smiling'), (-1, 'Male')]."""
    parsed = []

    for raw_attribute in query_text.split(","):
        raw_attribute = raw_attribute.strip()
        if not raw_attribute:
            continue

        if raw_attribute[0] not in "+-":
            raise ValueError(f"Query attribute must start with + or -: {raw_attribute!r}")

        sign = 1.0 if raw_attribute[0] == "+" else -1.0
        attribute = raw_attribute[1:].strip()
        parsed.append((sign, attribute))

    return parsed


def query_direction(query_index):
    """Turn a signed query into one CLIP text-direction vector on CPU."""
    if query_index in QUERY_DIRECTION_CACHE:
        return QUERY_DIRECTION_CACHE[query_index]

    direction = torch.zeros_like(IMAGE_EMBEDDINGS_NORMALIZED[0])

    for sign, attribute in parse_signed_query(annotations[query_index]["query"]):
        direction = direction + sign * text_embedding(attribute)

    QUERY_DIRECTION_CACHE[query_index] = direction
    return direction


def top_k_from_target_embedding(target_embedding, k, image_bank=None):
    """Rank dataset images by cosine similarity to a target embedding."""
    if image_bank is None:
        image_bank = IMAGE_EMBEDDINGS_NORMALIZED

    image_bank = image_bank.to(target_embedding.device)
    target_embedding = normalize_embeddings(target_embedding.float())
    similarities = image_bank @ target_embedding
    return similarities.argsort(descending=True)[:k].detach().cpu().tolist()


## 7. Zero-shot baselines

### Random baseline

This is only a sanity check for evaluation and visualization.

### Latent-space arithmetic

The query direction is added directly to the reference image embedding.

### SLERP

The reference image vector and query vector are combined by spherical interpolation. Use `alpha` to control how much the final vector moves toward the text direction.


In [ ]:
def random_predictions(case, k, seed=None):
    """Return k random image indices from the dataset."""
    if seed is None:
        return random.sample(range(len(image_embeddings)), k)

    rng = random.Random(f"{seed}-{case.query_index}-{case.reference_image_index}")
    return rng.sample(range(len(image_embeddings)), k)


def latent_arithmetic_predictions(case, k, text_scale=1.0):
    """Retrieve images using reference image embedding plus query text direction."""
    reference_embedding = IMAGE_EMBEDDINGS_NORMALIZED[int(case.reference_image_index)]
    target_embedding = reference_embedding + text_scale * query_direction(case.query_index)
    return top_k_from_target_embedding(target_embedding, k)


def slerp(v0, v1, t, eps=1e-8):
    """Spherical linear interpolation between two vectors."""
    v0 = normalize_embeddings(v0, eps=eps)
    v1 = normalize_embeddings(v1, eps=eps)

    dot = (v0 * v1).sum(dim=-1, keepdim=True).clamp(-1.0, 1.0)
    omega = torch.acos(dot)
    sin_omega = torch.sin(omega)

    if torch.any(sin_omega < eps):
        return normalize_embeddings((1.0 - t) * v0 + t * v1)

    return normalize_embeddings(
        torch.sin((1.0 - t) * omega) / sin_omega * v0
        + torch.sin(t * omega) / sin_omega * v1
    )


def slerp_predictions(case, k, alpha=0.5):
    """Retrieve images by SLERP-combining reference and query vectors."""
    reference_embedding = IMAGE_EMBEDDINGS_NORMALIZED[int(case.reference_image_index)]
    target_embedding = slerp(reference_embedding, query_direction(case.query_index), alpha)
    return top_k_from_target_embedding(target_embedding, k)


def run_random(cases, ks=KS, seed=0):
    return run_retrieval_method(
        cases,
        predict_fn=lambda case, k: random_predictions(case, k=k, seed=seed),
        method_name="Random baseline",
        ks=ks,
    )


def run_latent_arithmetic(cases, ks=KS, text_scale=1.0):
    return run_retrieval_method(
        cases,
        predict_fn=lambda case, k: latent_arithmetic_predictions(case, k=k, text_scale=text_scale),
        method_name="Latent arithmetic",
        ks=ks,
    )


def run_slerp(cases, ks=KS, alpha=0.5):
    return run_retrieval_method(
        cases,
        predict_fn=lambda case, k: slerp_predictions(case, k=k, alpha=alpha),
        method_name="SLERP",
        ks=ks,
    )


In [ ]:
# Smoke test on one small case.
display(reference_options(query_index=10, limit=10))

case = make_case(query_index=10, reference_position=1)

arithmetic_preds = latent_arithmetic_predictions(case, k=max(KS), text_scale=1.0)
inspect_case(case, arithmetic_preds, ks=KS, k_visualize=5)

slerp_preds = slerp_predictions(case, k=max(KS), alpha=0.5)
inspect_case(case, slerp_preds, ks=KS, k_visualize=5)


In [ ]:
# Limited zero-shot comparison. Increase max_references_per_query or set it to None for full runs.
small_cases = make_cases(query_indices=[10], max_references_per_query=20)

random_results = run_random(small_cases, ks=KS, seed=0)
arithmetic_results = run_latent_arithmetic(small_cases, ks=KS, text_scale=1.0)
slerp_results = run_slerp(small_cases, ks=KS, alpha=0.5)

compare_summaries({
    "random": random_results,
    "arithmetic": arithmetic_results,
    "slerp": slerp_results,
})


## 8. Shared training helpers for trainable retrieval methods

CoOp and NOAH both train small modules against frozen image embeddings. The generic training pair is:

```text
(reference image id, query index, positive target image id)
```

The loss compares the predicted target embedding against a candidate image pool that always contains the correct positive target plus sampled negatives.


In [ ]:
def build_training_pairs(cases, positives_per_case=1, seed=0):
    """Turn retrieval cases into reference/query/positive-target training pairs."""
    rng = random.Random(seed)
    pairs = []

    for case in cases:
        positives = list(case.ground_truth)

        if positives_per_case is None or positives_per_case >= len(positives):
            selected_positives = positives
        else:
            selected_positives = rng.sample(positives, positives_per_case)

        for target_image_index in selected_positives:
            pairs.append({
                "query_index": int(case.query_index),
                "reference_image_index": int(case.reference_image_index),
                "target_image_index": int(target_image_index),
            })

    return pairs


def candidate_pool_with_labels(positive_indices, bank_size, candidate_pool_size=None, device=DEVICE):
    """Build candidate image indices and labels for sampled-softmax training."""
    positive_indices = positive_indices.to(device)

    if candidate_pool_size is None or candidate_pool_size >= bank_size:
        candidate_indices = torch.arange(bank_size, device=device)
    else:
        positive_candidates = positive_indices.unique()
        num_negatives = max(int(candidate_pool_size) - len(positive_candidates), 0)
        negative_candidates = torch.randint(bank_size, (num_negatives,), device=device)
        candidate_indices = torch.cat([positive_candidates, negative_candidates]).unique()

    labels = (candidate_indices.unsqueeze(0) == positive_indices.unsqueeze(1)).float().argmax(dim=1)
    return candidate_indices, labels


def batch_query_directions(query_indices):
    """Stack cached zero-shot query directions for a batch."""
    return torch.stack([query_direction(int(query_index)) for query_index in query_indices])


## 9. CoOp: learned prompt context for retrieval

CoOp learns continuous prompt context tokens. CLIP itself stays frozen. For each attribute, the learned prompt has the form:

```text
[X1] [X2] [X3] [X4] Smiling.
```

The `X` tokens are trainable vectors, not literal words. After training, each attribute receives a task-adapted CLIP text vector. Query directions are still composed with signed addition/subtraction, but the attribute vectors now come from learned prompts.


In [ ]:
def coop_attribute_names():
    """Collect every unique attribute name used by the evaluation queries."""
    attributes = []
    seen = set()

    for query_index in range(len(annotations)):
        for _, attribute in parse_signed_query(annotations[query_index]["query"]):
            key = attribute.lower()
            if key not in seen:
                seen.add(key)
                attributes.append(attribute)

    return attributes


def coop_tokenizer():
    return preprocess.tokenizer if hasattr(preprocess, "tokenizer") else preprocess


def coop_max_text_length():
    return model.config.text_config.max_position_embeddings


def coop_expand_attention_mask(attention_mask, dtype):
    """Convert a 2D attention mask into the 4D mask expected by CLIP's encoder."""
    if attention_mask is None:
        return None

    expanded_mask = attention_mask[:, None, None, :].to(dtype=dtype)
    return (1.0 - expanded_mask) * torch.finfo(dtype).min


def coop_causal_attention_mask(batch_size, sequence_length, dtype, device):
    """Build CLIP's causal text-attention mask."""
    mask = torch.full(
        (batch_size, sequence_length, sequence_length),
        torch.finfo(dtype).min,
        dtype=dtype,
        device=device,
    )
    mask = torch.triu(mask, diagonal=1)
    return mask[:, None, :, :]


def coop_text_features_from_embeddings(input_embeddings, input_ids, attention_mask=None):
    """Run frozen HuggingFace CLIP text layers on custom prompt embeddings."""
    text_model = model.text_model
    text_device = next(text_model.parameters()).device
    text_dtype = next(text_model.parameters()).dtype

    input_embeddings = input_embeddings.to(device=text_device, dtype=text_dtype)
    input_ids = input_ids.to(text_device)
    attention_mask = None if attention_mask is None else attention_mask.to(text_device)

    batch_size, sequence_length, _ = input_embeddings.shape
    position_ids = torch.arange(sequence_length, device=text_device).unsqueeze(0)
    position_embeddings = text_model.embeddings.position_embedding(position_ids)
    hidden_states = input_embeddings + position_embeddings

    encoder_kwargs = dict(
        inputs_embeds=hidden_states,
        attention_mask=coop_expand_attention_mask(attention_mask, text_dtype),
        return_dict=True,
    )

    try:
        encoder_outputs = text_model.encoder(
            **encoder_kwargs,
            causal_attention_mask=coop_causal_attention_mask(batch_size, sequence_length, text_dtype, text_device),
        )
    except TypeError:
        encoder_outputs = text_model.encoder(**encoder_kwargs)

    last_hidden_state = text_model.final_layer_norm(encoder_outputs.last_hidden_state)
    pooled_state = last_hidden_state[torch.arange(batch_size, device=text_device), input_ids.argmax(dim=-1)]
    text_features = model.text_projection(pooled_state)

    return normalize_embeddings(text_features.float())


In [ ]:
class CoOpPromptLearner(nn.Module):
    """Unified-context CoOp prompt learner for CelebA attribute prompts."""

    def __init__(self, attribute_names, n_ctx=4, ctx_init="a photo of a"):
        super().__init__()

        self.attribute_names = list(attribute_names)
        self.attribute_to_index = {attribute.lower(): index for index, attribute in enumerate(self.attribute_names)}
        self.n_ctx = n_ctx

        tokenizer = coop_tokenizer()
        max_length = coop_max_text_length()
        prompt_prefix = " ".join(["X"] * n_ctx)
        prompt_texts = [f"{prompt_prefix} {attribute}." for attribute in self.attribute_names]

        tokenized = tokenizer(
            prompt_texts,
            padding="max_length",
            max_length=max_length,
            truncation=True,
            return_tensors="pt",
        )

        text_device = next(model.text_model.parameters()).device
        input_ids = tokenized["input_ids"].to(text_device)
        attention_mask = tokenized.get("attention_mask")
        attention_mask = None if attention_mask is None else attention_mask.to(text_device)

        with torch.no_grad():
            token_embeddings = model.text_model.embeddings.token_embedding(input_ids)
            ctx_vectors = self._initial_context_vectors(ctx_init, n_ctx, token_embeddings.dtype, text_device)

        self.ctx = nn.Parameter(ctx_vectors)
        self.register_buffer("input_ids", input_ids)

        if attention_mask is not None:
            self.register_buffer("attention_mask", attention_mask)
        else:
            self.attention_mask = None

        self.register_buffer("token_prefix", token_embeddings[:, :1, :])
        self.register_buffer("token_suffix", token_embeddings[:, 1 + n_ctx :, :])

    def _initial_context_vectors(self, ctx_init, n_ctx, dtype, device):
        if not ctx_init:
            ctx_vectors = torch.empty(n_ctx, model.config.text_config.hidden_size, dtype=dtype, device=device)
            nn.init.normal_(ctx_vectors, std=0.02)
            return ctx_vectors

        tokenizer = coop_tokenizer()
        tokenized = tokenizer(ctx_init, return_tensors="pt", padding=False, truncation=True)
        input_ids = tokenized["input_ids"].to(device)
        attention_mask = tokenized.get("attention_mask")

        with torch.no_grad():
            embeddings = model.text_model.embeddings.token_embedding(input_ids)[0]

        if attention_mask is not None:
            valid_length = int(attention_mask[0].sum().item())
            embeddings = embeddings[:valid_length]

        ctx_vectors = embeddings[1:-1]

        if len(ctx_vectors) >= n_ctx:
            return ctx_vectors[:n_ctx].clone().detach()

        padding = torch.empty(n_ctx - len(ctx_vectors), embeddings.shape[-1], dtype=dtype, device=device)
        nn.init.normal_(padding, std=0.02)
        return torch.cat([ctx_vectors, padding], dim=0).clone().detach()

    def forward(self):
        ctx = self.ctx.unsqueeze(0).expand(len(self.attribute_names), -1, -1)
        prompt_embeddings = torch.cat([self.token_prefix, ctx, self.token_suffix], dim=1)
        return coop_text_features_from_embeddings(prompt_embeddings, self.input_ids, self.attention_mask)

    def query_direction_from_features(self, query_index, attribute_features):
        direction = torch.zeros_like(attribute_features[0])

        for sign, attribute in parse_signed_query(annotations[query_index]["query"]):
            attribute_index = self.attribute_to_index[attribute.lower()]
            direction = direction + sign * attribute_features[attribute_index]

        return direction

    def query_direction(self, query_index):
        attribute_features = self.forward()
        return self.query_direction_from_features(query_index, attribute_features)


In [ ]:
def coop_batch_directions(prompt_learner, query_indices):
    """Compute learned query directions for a batch of query indices."""
    attribute_features = prompt_learner()
    return torch.stack([
        prompt_learner.query_direction_from_features(int(query_index), attribute_features)
        for query_index in query_indices
    ])


def train_coop_prompt(
    train_cases,
    n_ctx=4,
    ctx_init="a photo of a",
    epochs=3,
    batch_size=32,
    lr=2e-3,
    text_scale=1.0,
    positives_per_case=1,
    candidate_pool_size=512,
    seed=0,
):
    """Train CoOp context vectors while keeping CLIP and image embeddings frozen."""
    if not train_cases:
        raise ValueError("train_cases is empty. Build cases with make_cases(...) first.")

    torch.manual_seed(seed)
    random.seed(seed)

    prompt_learner = CoOpPromptLearner(
        attribute_names=coop_attribute_names(),
        n_ctx=n_ctx,
        ctx_init=ctx_init,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(prompt_learner.parameters(), lr=lr, weight_decay=0.0)
    image_bank = IMAGE_EMBEDDINGS_NORMALIZED.to(DEVICE)
    logit_scale = model.logit_scale.exp().detach().to(DEVICE) if hasattr(model, "logit_scale") else torch.tensor(1 / 0.07, device=DEVICE)
    pairs = build_training_pairs(train_cases, positives_per_case=positives_per_case, seed=seed)
    history = []

    for epoch in range(1, epochs + 1):
        random.shuffle(pairs)
        epoch_losses = []
        progress = tqdm(range(0, len(pairs), batch_size), desc=f"CoOp epoch {epoch}/{epochs}")

        for start in progress:
            batch = pairs[start : start + batch_size]
            query_indices = [item["query_index"] for item in batch]
            reference_indices = torch.tensor([item["reference_image_index"] for item in batch], device=DEVICE)
            positive_indices = torch.tensor([item["target_image_index"] for item in batch], device=DEVICE)

            reference_features = image_bank[reference_indices]
            query_directions = coop_batch_directions(prompt_learner, query_indices)
            target_features = normalize_embeddings(reference_features + text_scale * query_directions)

            candidate_indices, labels = candidate_pool_with_labels(
                positive_indices,
                bank_size=len(image_bank),
                candidate_pool_size=candidate_pool_size,
                device=DEVICE,
            )

            logits = logit_scale * target_features @ image_bank[candidate_indices].T
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
            progress.set_postfix(loss=sum(epoch_losses) / len(epoch_losses))

        history.append({"epoch": epoch, "loss": sum(epoch_losses) / max(len(epoch_losses), 1)})

    return prompt_learner, pd.DataFrame(history)


@torch.no_grad()
def coop_predictions(case, k, prompt_learner, text_scale=1.0):
    """Retrieve images using a trained CoOp prompt learner."""
    was_training = prompt_learner.training
    prompt_learner.eval()

    image_bank = IMAGE_EMBEDDINGS_NORMALIZED.to(DEVICE)
    reference_feature = image_bank[int(case.reference_image_index)]
    query_dir = prompt_learner.query_direction(case.query_index)
    target_feature = normalize_embeddings(reference_feature + text_scale * query_dir)
    predictions = top_k_from_target_embedding(target_feature, k, image_bank=image_bank)

    if was_training:
        prompt_learner.train()

    return predictions


def run_coop(cases, prompt_learner, ks=KS, text_scale=1.0):
    return run_retrieval_method(
        cases,
        predict_fn=lambda case, k: coop_predictions(case, k=k, prompt_learner=prompt_learner, text_scale=text_scale),
        method_name="CoOp retrieval",
        ks=ks,
    )


In [ ]:
# Small CoOp training run. Start here before attempting larger training.
coop_train_cases = make_cases(query_indices=[10], max_references_per_query=20)
coop_prompt_learner, coop_history = train_coop_prompt(
    coop_train_cases,
    n_ctx=4,
    ctx_init="a photo of a",
    epochs=3,
    batch_size=16,
    lr=2e-3,
    text_scale=1.0,
    positives_per_case=1,
    candidate_pool_size=512,
    seed=SEED,
)
coop_history


In [ ]:
# Limited CoOp evaluation.
coop_eval_cases = make_cases(query_indices=[10], max_references_per_query=20)
coop_results = run_coop(coop_eval_cases, prompt_learner=coop_prompt_learner, ks=KS, text_scale=1.0)
summarize_results(coop_results)


## 10. NOAH-style searchable residual adapters

The previous NOAH section had the right architectural idea but used placeholder training/search/evaluation code. This section completes the loop.

The NOAH-style model has three residual bottleneck adapters:

1. `reference_adapter`: adapts the frozen CLIP image embedding of the reference image;
2. `query_adapter`: adapts the CLIP query direction;
3. `target_adapter`: adapts the fused retrieval vector.

A candidate architecture chooses:

```python
reference_adapter_dim
query_adapter_dim
target_adapter_dim
text_scale
```

Adapter dimension `0` means the corresponding adapter is skipped. This creates a small search space over parameter-efficient retrieval adaptations.


In [ ]:
class NoahResidualAdapter(nn.Module):
    """Residual bottleneck adapter with a searchable active bottleneck width."""

    def __init__(self, embedding_dim, max_bottleneck=256):
        super().__init__()
        self.max_bottleneck = max_bottleneck
        self.down = nn.Linear(embedding_dim, max_bottleneck, bias=False)
        self.up = nn.Linear(max_bottleneck, embedding_dim, bias=False)

        # Small initialization keeps early behavior close to identity.
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)

    def forward(self, x, active_dim):
        if active_dim == 0:
            return x

        if active_dim < 0 or active_dim > self.max_bottleneck:
            raise ValueError(f"active_dim must be in [0, {self.max_bottleneck}], got {active_dim}")

        z = self.down(x)[..., :active_dim]
        out = F.linear(z, self.up.weight[:, :active_dim])
        return x + out


class NoahSupernet(nn.Module):
    """Searchable retrieval adapter supernet inspired by NOAH."""

    def __init__(self, embedding_dim, max_adapter_dim=256):
        super().__init__()
        self.reference_adapter = NoahResidualAdapter(embedding_dim, max_adapter_dim)
        self.query_adapter = NoahResidualAdapter(embedding_dim, max_adapter_dim)
        self.target_adapter = NoahResidualAdapter(embedding_dim, max_adapter_dim)

    def forward(self, reference_embedding, query_direction_embedding, candidate):
        ref = self.reference_adapter(reference_embedding, candidate["reference_adapter_dim"])
        q = self.query_adapter(query_direction_embedding, candidate["query_adapter_dim"])

        fused = normalize_embeddings(ref + candidate["text_scale"] * q)
        target = self.target_adapter(fused, candidate["target_adapter_dim"])
        return normalize_embeddings(target)


def sample_noah_candidate(
    adapter_dims=(0, 8, 16, 32, 64, 128),
    text_scales=(0.25, 0.5, 0.75, 1.0, 1.25),
):
    """Sample one NOAH-style candidate architecture."""
    return {
        "reference_adapter_dim": random.choice(adapter_dims),
        "query_adapter_dim": random.choice(adapter_dims),
        "target_adapter_dim": random.choice(adapter_dims),
        "text_scale": random.choice(text_scales),
    }


def canonical_noah_candidate():
    """A conservative default candidate for smoke tests."""
    return {
        "reference_adapter_dim": 32,
        "query_adapter_dim": 32,
        "target_adapter_dim": 32,
        "text_scale": 1.0,
    }


In [ ]:
def noah_batch_directions(query_indices):
    """Stack zero-shot CLIP query directions for NOAH training/evaluation."""
    return batch_query_directions(query_indices).to(DEVICE)


def train_noah_supernet(
    noah_model,
    train_cases,
    epochs=3,
    batch_size=32,
    lr=1e-3,
    positives_per_case=1,
    candidate_pool_size=512,
    adapter_dims=(0, 8, 16, 32, 64, 128),
    text_scales=(0.25, 0.5, 0.75, 1.0, 1.25),
    fixed_candidate=None,
    seed=0,
):
    """
    Train the NOAH supernet with real retrieval supervision.

    If fixed_candidate is None, a new candidate is sampled for each batch, approximating
    one-shot supernet training. If fixed_candidate is provided, only that architecture
    is trained/evaluated.
    """
    if not train_cases:
        raise ValueError("train_cases is empty. Build cases with make_cases(...) first.")

    torch.manual_seed(seed)
    random.seed(seed)

    noah_model = noah_model.to(DEVICE)
    noah_model.train()

    optimizer = torch.optim.AdamW(noah_model.parameters(), lr=lr, weight_decay=1e-4)
    image_bank = IMAGE_EMBEDDINGS_NORMALIZED.to(DEVICE)
    logit_scale = model.logit_scale.exp().detach().to(DEVICE) if hasattr(model, "logit_scale") else torch.tensor(1 / 0.07, device=DEVICE)
    pairs = build_training_pairs(train_cases, positives_per_case=positives_per_case, seed=seed)
    history = []

    for epoch in range(1, epochs + 1):
        random.shuffle(pairs)
        epoch_losses = []
        progress = tqdm(range(0, len(pairs), batch_size), desc=f"NOAH epoch {epoch}/{epochs}")

        for start in progress:
            batch = pairs[start : start + batch_size]
            query_indices = [item["query_index"] for item in batch]
            reference_indices = torch.tensor([item["reference_image_index"] for item in batch], device=DEVICE)
            positive_indices = torch.tensor([item["target_image_index"] for item in batch], device=DEVICE)

            candidate = fixed_candidate or sample_noah_candidate(adapter_dims=adapter_dims, text_scales=text_scales)
            reference_features = image_bank[reference_indices]
            query_features = noah_batch_directions(query_indices)
            target_features = noah_model(reference_features, query_features, candidate)

            candidate_indices, labels = candidate_pool_with_labels(
                positive_indices,
                bank_size=len(image_bank),
                candidate_pool_size=candidate_pool_size,
                device=DEVICE,
            )

            logits = logit_scale * target_features @ image_bank[candidate_indices].T
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
            progress.set_postfix(loss=sum(epoch_losses) / len(epoch_losses))

        history.append({"epoch": epoch, "loss": sum(epoch_losses) / max(len(epoch_losses), 1)})

    return noah_model, pd.DataFrame(history)


In [ ]:
@torch.no_grad()
def noah_target_embedding(case, noah_model, candidate):
    """Build the adapted target retrieval embedding for one case."""
    was_training = noah_model.training
    noah_model.eval()

    image_bank = IMAGE_EMBEDDINGS_NORMALIZED.to(DEVICE)
    reference_feature = image_bank[int(case.reference_image_index)].unsqueeze(0)
    query_feature = query_direction(case.query_index).to(DEVICE).unsqueeze(0)
    target_feature = noah_model(reference_feature, query_feature, candidate).squeeze(0)

    if was_training:
        noah_model.train()

    return target_feature


@torch.no_grad()
def noah_predictions(case, k, noah_model, candidate):
    """Retrieve top-k images with a trained NOAH model and a selected candidate."""
    image_bank = IMAGE_EMBEDDINGS_NORMALIZED.to(DEVICE)
    target_feature = noah_target_embedding(case, noah_model, candidate)
    return top_k_from_target_embedding(target_feature, k, image_bank=image_bank)


def run_noah(cases, noah_model, candidate, ks=KS):
    """Complete NOAH evaluation loop for selected cases."""
    return run_retrieval_method(
        cases,
        predict_fn=lambda case, k: noah_predictions(case, k=k, noah_model=noah_model, candidate=candidate),
        method_name="NOAH retrieval",
        ks=ks,
    )


def noah_candidate_score(results, score_metric="Recall@10"):
    """Score a candidate from its per-case result rows."""
    if not results:
        return float("-inf")

    df = pd.DataFrame(results)
    if score_metric not in df.columns:
        raise ValueError(f"Metric {score_metric!r} not found. Available columns: {list(df.columns)}")

    return float(df[score_metric].mean())


def search_noah_candidates(
    noah_model,
    validation_cases,
    num_candidates=20,
    adapter_dims=(0, 8, 16, 32, 64, 128),
    text_scales=(0.25, 0.5, 0.75, 1.0, 1.25),
    ks=KS,
    score_metric="Recall@10",
    seed=0,
):
    """
    Evaluate sampled NOAH candidates on validation cases and return a ranked table.

    This replaces the previous placeholder random search: every candidate is scored by
    actual retrieval predictions and the unchanged evaluate_retrieval metric.
    """
    if not validation_cases:
        raise ValueError("validation_cases is empty. Use split_cases(...) with enough cases.")

    rng_state = random.getstate()
    random.seed(seed)

    rows = []
    best_candidate = None
    best_score = float("-inf")

    for candidate_id in tqdm(range(num_candidates), desc="Searching NOAH candidates"):
        candidate = sample_noah_candidate(adapter_dims=adapter_dims, text_scales=text_scales)
        results = run_noah(validation_cases, noah_model=noah_model, candidate=candidate, ks=ks)
        score = noah_candidate_score(results, score_metric=score_metric)

        row = {"candidate_id": candidate_id, **candidate, "score_metric": score_metric, "score": score}
        rows.append(row)

        if score > best_score:
            best_score = score
            best_candidate = candidate

    random.setstate(rng_state)

    search_table = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    return search_table, best_candidate


In [ ]:
# Limited NOAH training/search/evaluation example.
# This is intentionally small; increase the limits for serious experiments.
noah_cases = make_cases(query_indices=[10], max_references_per_query=40)
noah_train_cases, noah_val_cases = split_cases(noah_cases, validation_fraction=0.25, seed=SEED)

noah_supernet = NoahSupernet(embedding_dim=EMBEDDING_DIM, max_adapter_dim=128).to(DEVICE)
noah_supernet, noah_history = train_noah_supernet(
    noah_supernet,
    train_cases=noah_train_cases,
    epochs=3,
    batch_size=16,
    lr=1e-3,
    positives_per_case=1,
    candidate_pool_size=512,
    adapter_dims=(0, 8, 16, 32, 64, 128),
    text_scales=(0.5, 0.75, 1.0, 1.25),
    fixed_candidate=None,
    seed=SEED,
)

noah_history


In [ ]:
# Complete NOAH candidate evaluation loop.
noah_search_table, best_noah_candidate = search_noah_candidates(
    noah_supernet,
    validation_cases=noah_val_cases,
    num_candidates=10,
    adapter_dims=(0, 8, 16, 32, 64, 128),
    text_scales=(0.5, 0.75, 1.0, 1.25),
    ks=KS,
    score_metric="Recall@10",
    seed=SEED,
)

display(noah_search_table)
print("Best NOAH candidate:", best_noah_candidate)


In [ ]:
# Evaluate the selected NOAH candidate on the validation split.
noah_results = run_noah(noah_val_cases, noah_model=noah_supernet, candidate=best_noah_candidate, ks=KS)
summarize_results(noah_results)


## 11. Final experiment templates

The cells below are templates for larger runs. They are commented out by default because full evaluation over all query/reference pairs can be slow.

Suggested workflow:

1. run the zero-shot methods on a limited set of references;
2. train CoOp on a limited set and verify that loss decreases;
3. train NOAH on a limited set and search candidates on validation cases;
4. only then run the full evaluation.


In [ ]:
# Full zero-shot evaluation template.
# full_cases = make_cases(query_indices=None, max_references_per_query=None)
# full_arithmetic_results = run_latent_arithmetic(full_cases, ks=KS, text_scale=1.0)
# full_slerp_results = run_slerp(full_cases, ks=KS, alpha=0.5)
# compare_summaries({
#     "arithmetic": full_arithmetic_results,
#     "slerp": full_slerp_results,
# })


In [ ]:
# Larger CoOp training/evaluation template.
# coop_train_cases = make_cases(query_indices=None, max_references_per_query=100)
# coop_prompt_learner, coop_history = train_coop_prompt(
#     coop_train_cases,
#     n_ctx=4,
#     ctx_init="a photo of a",
#     epochs=5,
#     batch_size=32,
#     lr=2e-3,
#     text_scale=1.0,
#     positives_per_case=1,
#     candidate_pool_size=1024,
#     seed=SEED,
# )
# coop_eval_cases = make_cases(query_indices=None, max_references_per_query=None)
# coop_results = run_coop(coop_eval_cases, prompt_learner=coop_prompt_learner, ks=KS, text_scale=1.0)
# summarize_results(coop_results)


In [ ]:
# Larger NOAH training/search/evaluation template.
# noah_all_cases = make_cases(query_indices=None, max_references_per_query=100)
# noah_train_cases, noah_val_cases = split_cases(noah_all_cases, validation_fraction=0.2, seed=SEED)
#
# noah_supernet = NoahSupernet(embedding_dim=EMBEDDING_DIM, max_adapter_dim=128).to(DEVICE)
# noah_supernet, noah_history = train_noah_supernet(
#     noah_supernet,
#     train_cases=noah_train_cases,
#     epochs=5,
#     batch_size=32,
#     lr=1e-3,
#     positives_per_case=1,
#     candidate_pool_size=1024,
#     adapter_dims=(0, 8, 16, 32, 64, 128),
#     text_scales=(0.5, 0.75, 1.0, 1.25),
#     fixed_candidate=None,
#     seed=SEED,
# )
#
# noah_search_table, best_noah_candidate = search_noah_candidates(
#     noah_supernet,
#     validation_cases=noah_val_cases,
#     num_candidates=25,
#     adapter_dims=(0, 8, 16, 32, 64, 128),
#     text_scales=(0.5, 0.75, 1.0, 1.25),
#     ks=KS,
#     score_metric="Recall@10",
#     seed=SEED,
# )
#
# final_noah_cases = make_cases(query_indices=None, max_references_per_query=None)
# final_noah_results = run_noah(final_noah_cases, noah_model=noah_supernet, candidate=best_noah_candidate, ks=KS)
# summarize_results(final_noah_results)


In [ ]:
# One-table comparison template after you have produced result lists.
# compare_summaries({
#     "arithmetic": full_arithmetic_results,
#     "slerp": full_slerp_results,
#     "coop": coop_results,
#     "noah": final_noah_results,
# })
